# Stat 220, Unit 8 Homework: Probability as a Model

Much of this is pencil work. For parts marked **by hand**, work out the calculation
before using the code cell to check it. Please enter the formula you used rather than typing
the answer directly. For example, enter the multiplication that gives the answer and let
Python calculate the result.

The first five problems use small examples or model information given in the question.
The last two use the 700 lawn mowing jobs from Unit 5 and the 500 delivery routes from the
Unit 4 and Unit 6 homeworks. You will estimate probabilities from those data and compare
them with what a fitted model predicts.

Run the next two cells first.

In [2]:
%pip install -q numpy pandas scipy matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binom, poisson, expon, norm, lognorm

lawn = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/lawn_jobs.csv")
routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
lawn.head()

,minutes,lot_sqft,slope,obstacles,gated,crew_size,invoice_id,customer_rating
0,13.3,11125,flat,3,0,3,50761,5
1,4.5,1362,moderate,4,0,3,68868,4
2,14.4,5919,flat,3,0,1,89918,2
3,32.8,15539,flat,3,1,1,64082,4
4,6.8,3532,flat,3,0,3,82748,3


For a discrete distribution, `pmf(k, ...)` gives the probability of exactly `k`, and `sf(k, ...)` gives the probability of more than `k`. For a continuous distribution, `cdf(x, ...)` gives the area below `x`, `sf(x, ...)` gives the area above it, and `ppf(q, ...)` gives the value with proportion `q` below it.

**Problem 1.** Of the 700 lawn mowing jobs, 144 were at properties with a gate.
Let $p$ be the probability that a job is at a gated property. For now, assume that new
bookings are independent and have the same gate probability as these jobs.

Part a. **By hand.** Estimate $p$ using the counts above. Write the fraction and
calculate its value. Then check it using the `gated` column in `lawn`, where 1 means the
property has a gate and 0 means it does not.

In [3]:
# Your Answer
p_hat = 144 / 700
print("Manual fraction:", p_hat)
print("Data check:", lawn['gated'].mean())

Manual fraction: 0.2057142857142857
Data check: 0.2057142857142857


_Your answer:_

p^ = 144/700 = 36/175 ≈ 0.2057

Part b. **By hand.** Twenty-five jobs are booked for next week. Using your estimate
from part a, give the mean and standard deviation of the number of gated jobs. Show the
variance calculation before taking the square root.

In [4]:
# Your Answer
n = 25
p_hat = 144 / 700

mean_b = n * p_hat
var_b = n * p_hat * (1 - p_hat)
sd_b = np.sqrt(var_b)

print("Mean:", mean_b)
print("Variance:", var_b)
print("Standard Deviation:", sd_b)

Mean: 5.142857142857142
Variance: 4.084897959183673
Standard Deviation: 2.0211130495802734


_Your answer:_

Mean = μ = n * p = 25 * 144/700 = 3600/700 ≈ 5.1429

Variance = σ^2 = n * p * (1−p) = 25 * 144/700 * (1 − 144/700) = 25 * 144/700 × 556/700 ≈ 4.0849

Standard Deviation = σ = SQRT(4.0849) ≈ 2.0211

Part c. Use `binom` to find the probability that exactly 5 of the 25 jobs are gated,
then the probability that 8 or more are gated. Which probability would be more useful if
the company needed extra equipment whenever it had at least 8 gated jobs?

In [5]:
# Your Answer
n = 25
p_hat = 144 / 700

prob_exact_5 = binom.pmf(5, n, p_hat)
prob_8_or_more = binom.sf(7, n, p_hat)

print("P(X = 5):", prob_exact_5)
print("P(X >= 8):", prob_8_or_more)

P(X = 5): 0.19552266480920347
P(X >= 8): 0.12393234209954235


Part d. **By hand.** Calculate the standard error of your estimate of $p$,
$\widehat{\mathrm{SE}}(\hat p)=\sqrt{\hat p(1-\hat p)/700}$, and an approximate 95% confidence
interval, $\hat p\pm1.96\,\widehat{\mathrm{SE}}(\hat p)$. Explain how this standard error
differs from the standard deviation in part b.

In [6]:
# Your Answer
n_sample = 700
p_hat = 144 / 700

se = np.sqrt((p_hat * (1 - p_hat)) / n_sample)
ci_low = p_hat - 1.96 * se
ci_high = p_hat + 1.96 * se

print("Standard Error:", se)
print("95% CI:", (ci_low, ci_high))

Standard Error: 0.015278178573533604
95% CI: (np.float64(0.17576905571015985), np.float64(0.23565951571841157))


_Your answer:_

Standard Error = SE (p^) = SQRT(p^ (1 − p^) / 700) = SQRT(144/700 * 556/700) / 700 = SQRT(0.205714 * 0.794286) / 700 = SQRT(0.0002334) ≈ 0.0153

95% CI = p^ ± 1.96 * SE (p^) = 0.2057 ± 1.96 * (0.0153) = 0.2057 ± 0.0299 ⟹ (0.1758, 0.2356)

Explain: The standard error in part d measures the uncertainty in our estimate of the true population proportion (p^) based on our sample of 700 historical jobs, which tells us how much p^ would vary from sample to sample. In contrast, the standard deviation in part b measures the natural random variability in the actual count of gated jobs (X) we expect to see in a single future week of 25 jobs.

Part e. **By hand.** The office now knows that 10 of next week's 25 jobs are gated.
Only the remaining 15 are uncertain; model their gate indicators as independent with your
estimated probability. What are the new mean and standard deviation of the total number
of gated jobs? Explain why the 10 known jobs add nothing to its variance.

In [7]:
# Your Answer
n_uncertain = 15
p_hat = 144 / 700

new_mean = 10 + (n_uncertain * p_hat)
new_var = n_uncertain * p_hat * (1 - p_hat)
new_sd = np.sqrt(new_var)

print("New Mean:", new_mean)
print("New Standard Deviation:", new_sd)

New Mean: 13.085714285714285
New Standard Deviation: 1.565547436365377


_Your answer:_

New Mean = μT = 10 + (15 * 144/700) = 10 + 3.0857 = 13.0857

New Variance = σ^2T = 0 + 15 * 144/700 * 556/700 ≈ 2.4509

New Standard Deviation = σT = SQRT(2.4509) ≈ 1.5655

Explain: The 10 known jobs add nothing to the variance because they are constants, not random variables. Because their outcomes are already certain (100% chance of having a gate), they have zero variability (Var(10) = 0) and only shift the distribution's center (mean) upward by 10 without spreading it out.

**Problem 2.** A smaller example will help us see how likelihood works. The office sends
10 quotes, and 3 become bookings. Assume independent responses with a common booking
probability $p$. We will use the likelihood of the individual responses.

Part a. **By hand.** Write the likelihood $\mathcal L(p)$ for these responses.
Fill in the exponents in $p^{\square}(1-p)^{\square}$ and explain what each exponent counts.

_Your answer:_



Part b. **By hand.** Evaluate the likelihood at $p=0.20$, $p=0.30$, and $p=0.50$.
Show the powers and multiplication in your code cell. Which of these values gives the
largest likelihood? How does it compare with the observed booking proportion?

In [ ]:
# Your Answer

_Your answer:_



Part c. Now calculate $\log\mathcal L(p)$ for the same three values, using
$3\log(p)+7\log(1-p)$. Does taking the log change which value fits best? In two sentences,
explain why software uses log-likelihood and why the highest likelihood does not mean
there is a high probability that $p$ equals that value.

In [ ]:
# Your Answer

_Your answer:_



**Problem 3.** The lawn company's office receives cancellation calls at an average rate
of 2 per hour. Suppose the calls follow a Poisson process with a constant rate. The count
over a time period is Poisson, and the waiting time between calls is exponential.

Part a. **By hand.** Give the mean, variance, and standard deviation of the number
of calls in one hour. Then do the same for a three-hour period. Keep track of the time
period that goes with each Poisson parameter.

In [ ]:
# Your Answer

Part b. **By hand.** Find the probability of no calls in the next hour using
$\mathbb P(X=0)=e^{-\lambda}$. Check it with `poisson.pmf`. Then use `poisson.sf` to find
the probability of 4 or more calls in that hour.

In [ ]:
# Your Answer

Part c. **By hand.** What is the mean wait between calls, in minutes? Use
$\mathbb P(T>t)=e^{-t/30}$ to find the chance of waiting more than 15 minutes and more
than 30 minutes. Check your answers with `expon.sf(t, scale=30)`. Why is the probability
of waiting longer than the mean not 0.50?

In [ ]:
# Your Answer

_Your answer:_



Part d. No calls have arrived for 30 minutes. Under this model, what is the chance
of waiting at least another 15 minutes? Explain which property of the exponential
distribution you used. Would a higher cancellation rate on rainy mornings make one
constant rate a reasonable model for all mornings?

In [ ]:
# Your Answer

_Your answer:_



**Problem 4.** For one delivery route, a model predicts 55 minutes and gives a 95%
prediction interval of 40 to 70 minutes. Treat the forecast as normal and use 1.96 as the
critical value. We will use this forecast to make a delivery promise.

Part a. **By hand.** Recover $\mathrm{SE}_{\mathrm{pred}}$ from the interval.
Show the width divided by $2\times1.96$. Store the result as `se_pred` for the next parts.

In [ ]:
# Your Answer

Part b. **By hand.** Find the z-score for 60 minutes. Then use `norm.sf` to find
the chance of missing a 60-minute delivery promise, and `norm.cdf` to find the chance
of finishing between 45 and 65 minutes.

In [ ]:
# Your Answer

Part c. What delivery time would allow 90% of similar routes to finish by the
promised time? Use `norm.ppf`. Explain why the point prediction of 55 minutes does not
provide the same level of coverage.

In [ ]:
# Your Answer

_Your answer:_



Part d. **By hand.** Suppose three independent routes have this same forecast.
Give the mean and standard deviation of their total time, showing the variance step.
Then use the normal distribution to find the chance that the total exceeds 180 minutes.
Name one reason route times might not be independent in practice.

In [ ]:
# Your Answer

_Your answer:_



**Problem 5.** For a particular lawn, a model describes $\log(Y)$ as normal with mean
3 and standard deviation 0.4, where $Y$ is the job time in minutes. Thus the forecast for
$Y$ is lognormal. Use these given parameters throughout this problem.

Part a. **By hand.** Calculate the median time, $e^3$, and the mean time,
$e^{3+0.4^2/2}$. Explain why exponentiating the predicted log time gives the median.

In [ ]:
# Your Answer

_Your answer:_



Part b. **By hand.** Find a 95% interval on the log scale using
$3\pm1.96(0.4)$, then exponentiate the endpoints to get a prediction interval in minutes.
How far is each endpoint from the median? Explain why those distances differ.

In [ ]:
# Your Answer

_Your answer:_



Part c. Use `lognorm` to find the probability that this job takes more than 30
minutes and the time that 90% of similar jobs finish within. Use `s=0.4` and
`scale=np.exp(3)`. Compare the 90th percentile with the median from part a.

In [ ]:
# Your Answer

_Your answer:_



Part d. The company's labor cost is \$1.10 per minute. It also gives the customer
a \$25 credit if the job takes more than 30 minutes. First calculate the cost using only
the median time. Then calculate the expected cost,

$$1.10\,\mathbb E[Y]+25\,\mathbb P(Y>30).$$

What would you budget for 400 similar jobs? How much larger is that than a budget based
only on the median? Explain which possible outcomes the median-only calculation leaves out.

In [ ]:
# Your Answer

_Your answer:_



**Problem 6.** Now use the data to check some Poisson models. `lawn.obstacles` counts
trees, garden beds, and other obstacles at each property. In `routes`, `packages` counts
parcels on a route and `van_age_years` records the van's age in whole years.
Use the sample variance, as returned by `.var()`.

Part a. For `obstacles`, estimate $\lambda$ with the sample mean. Calculate the
sample variance and the variance-to-mean ratio. What relationship would you expect under
a Poisson model, and how close are these data to it?

In [ ]:
# Your Answer

_Your answer:_



Part b. Using your fitted Poisson model, find the probability of no obstacles and
of 6 or more obstacles. Put each beside its observed proportion in the data. For example,
`(lawn.obstacles == 0).mean()` gives the observed proportion with no obstacles.
Do these comparisons support using a Poisson model? Explain briefly.

In [ ]:
# Your Answer

_Your answer:_



Part c. For `routes.packages`, calculate the mean, variance, and variance-to-mean
ratio. Compare the fitted Poisson probability of more than 200 parcels with the observed
proportion. What is overdispersion, and what could a dispatcher underestimate by using
this Poisson model?

In [ ]:
# Your Answer

_Your answer:_



Part d. For `routes.van_age_years`, calculate the mean, variance, and their ratio.
Then compare the observed numbers of vans aged 0 and 8 with the expected numbers under
the fitted Poisson model. An expected number is `len(routes) * poisson.pmf(k, lam)`.
Is a variance-to-mean ratio near 1 enough to conclude that the model fits? Use your
results to explain.

In [ ]:
# Your Answer

_Your answer:_



**Problem 7.** Finally, compare normal and lognormal models for `lawn.minutes`.
We are modeling the times across all properties in the dataset, rather than making
a forecast for one particular property.

Part a. Calculate the mean, standard deviation, and skewness of `minutes`.
Fit a normal model using the mean and SD, and compare its probability of a job taking
more than 30 minutes with the observed proportion. How close are the two probabilities?

In [ ]:
# Your Answer

_Your answer:_



Part b. What probability does the same normal model assign to a job taking less
than zero minutes? Why is that a concern here? Discuss it alongside the comparison in
part a when deciding whether this model is useful for scheduling.

In [ ]:
# Your Answer

_Your answer:_



Part c. Run these Q-Q plots, which compare `minutes` and `log(minutes)` with a
normal distribution. Describe the main pattern in each plot. Which scale looks closer
to normal?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
stats.probplot(lawn.minutes, dist="norm", plot=axes[0])
axes[0].set_title("Job time in minutes")
stats.probplot(np.log(lawn.minutes), dist="norm", plot=axes[1])
axes[1].set_title("Log of job time")
plt.tight_layout()
plt.show()

_Your answer:_



Part d. Calculate the mean and SD of `np.log(lawn.minutes)`. Use them as `mu_log`
and `sigma_log` in a lognormal model, then find its probability of a job taking more
than 30 minutes using `lognorm.sf(30, s=sigma_log, scale=np.exp(mu_log))`.
Compare this with part a and the observed proportion. Which model would you recommend
for estimating this probability, and what evidence supports your choice?

In [ ]:
# Your Answer

_Your answer:_

